# PQC Migration Lab Analysis

This notebook analyzes the synthetic cryptographic asset inventory and benchmark results
generated by `src/generate_data.py`.

**Lab:** qc-pqc-migration-lab  
**Standard:** NIST SP 800-208 / CNSA 2.0  
**Generated:** 2026-10-06

## Contents
1. Crypto Asset Inventory — distribution by algorithm and migration status
2. Algorithm Benchmark Comparison — latency and key/signature sizes
3. HNDL Exposure Analysis — assets at risk from Harvest-Now-Decrypt-Later
4. Migration Phase Summary

In [ ]:
import sys
import json
import pandas as pd
from pathlib import Path

# Allow importing from src/
sys.path.insert(0, str(Path().resolve().parent / 'src'))

DATA_DIR = Path().resolve().parent / 'data'
print('Data directory:', DATA_DIR)
print('Files present:', [f.name for f in DATA_DIR.iterdir()] if DATA_DIR.exists() else 'Run src/generate_data.py first')

## Crypto Asset Inventory

The CBOM (Cryptographic Bill of Materials) contains 50 enterprise assets.
We analyse distribution by algorithm and migration status.

In [ ]:
df = pd.read_csv(DATA_DIR / 'crypto_assets.csv')
print(f'Total assets: {len(df)}')
print()
print('Assets by algorithm:')
print(df.groupby('algorithm')['asset_id'].count().to_string())
print()
print('Assets by migration status:')
print(df.groupby('migration_status')['asset_id'].count().to_string())
print()
print('HNDL exposure by algorithm:')
print(df.groupby(['algorithm', 'hndl_exposure'])['asset_id'].count().to_string())
print()
print('Risk level distribution:')
print(df.groupby('risk_level')['asset_id'].count().to_string())

## Algorithm Benchmark Comparison

Side-by-side latency comparison between classical (RSA, ECDSA) and
post-quantum (ML-KEM-768, ML-DSA-65, SLH-DSA-128f) algorithms.

Key insight: ML-KEM-768 keygen/encap is **10x faster** than RSA-2048 keygen.
SLH-DSA-128f signing at 35 ms is the performance outlier — use only for code signing.

In [ ]:
df_bench = pd.read_csv(DATA_DIR / 'benchmark_results.csv')
print('Algorithm Benchmark Results:')
print(df_bench[['algorithm', 'operation', 'latency_ms', 'quantum_safe']].to_string(index=False))
print()
print('Key and signature sizes (bytes):')
print(df_bench[['algorithm', 'operation', 'key_size_bytes', 'sig_size_bytes']].to_string(index=False))
print()
# Signing latency comparison
sign_df = df_bench[df_bench['operation'] == 'sign'][['algorithm', 'latency_ms', 'quantum_safe']]
sign_df = sign_df.sort_values('latency_ms')
print('Sign operation latency (sorted):')
print(sign_df.to_string(index=False))

## HNDL Exposure Analysis

HNDL (Harvest Now, Decrypt Later) is the primary near-term quantum threat.
Adversaries collect encrypted traffic today for decryption once a CRQC is available.
CNSA 2.0 requires mitigation of HNDL-exposed systems by 2026.

In [ ]:
hndl_exposed = df[df['hndl_exposure'] == 'true']
print(f'Total HNDL-exposed assets: {len(hndl_exposed)}')
print()
print('HNDL-exposed assets by location:')
print(hndl_exposed.groupby('location')['asset_id'].count().sort_values(ascending=False).to_string())
print()
print('HNDL-exposed assets by migration status:')
print(hndl_exposed.groupby('migration_status')['asset_id'].count().to_string())
print()
not_started_hndl = hndl_exposed[hndl_exposed['migration_status'] == 'not-started']
print(f'CRITICAL: {len(not_started_hndl)} HNDL-exposed assets with migration not yet started')

## Migration Phase Summary

The CNSA 2.0 migration is divided into 4 phases: Discovery, Hybrid Mode, PQC Primary, and Classical Deprecation.

In [ ]:
timeline_path = DATA_DIR / 'migration_timeline.json'
with open(timeline_path) as f:
    timeline = json.load(f)

print(f"Standard: {timeline['standard']}")
print(f"Deadline: {timeline['deadline']}")
print()
print('Phase Summary:')
print(f'{"Phase":<5} {"Name":<30} {"Period":<25} {"Status":<15}')
print('-' * 75)
for p in timeline['phases']:
    period = f"{p['start']} → {p['end']}"
    print(f"{p['phase']:<5} {p['name']:<30} {period:<25} {p['status']:<15}")
print()
print('Protocol Migration Targets:')
for proto, targets in timeline['protocol_timeline'].items():
    print(f"  {proto:<15} Hybrid: {targets['target_hybrid']}  |  PQC Primary: {targets['target_pqc_primary']}")